In [6]:
!rm -r /kaggle/working/RCS_FUSIONDATA
!git clone https://github.com/Dlevinh755/RCS_FUSIONDATA.git

Cloning into 'RCS_FUSIONDATA'...
remote: Enumerating objects: 39156, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 39156 (delta 64), reused 100 (delta 35), pack-reused 39024 (from 3)
Receiving objects: 100% (39156/39156), 387.02 MiB | 41.61 MiB/s, done.
Resolving deltas: 100% (210/210), done.
Updating files: 100% (38826/38826), done.


In [ ]:
%%capture
!pip install -r /kaggle/working/RCS_FUSIONDATA/requirements.txt

In [ ]:
# !python /kaggle/working/RCS_FUSIONDATA/prepare_data/prepare_amazon_product.py \
# --meta_link https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/metaFiles2/meta_Appliances.json.gz \
# --reviews_link https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/categoryFiles/Appliances.json.gz \
# --mode None \
# --json-parser parallel

In [ ]:
!cd /kaggle/working/RCS_FUSIONDATA
!python /kaggle/working/RCS_FUSIONDATA/main.py \
--epochs 15 \
--batch_size 256 \
--lr 1e-4 \
--patience 3 

⚡ Memory-efficient mode - processing on-the-fly
⚡ Memory-efficient mode - processing on-the-fly
⚡ Memory-efficient mode - processing on-the-fly
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
🚀 Training CAMRec | Trainable params: 6,819,969
Epoch 1/15: 100%|██████████████████| 98/98 [02:46<00:00,  1.70s/it, Loss=1.3397]
Epoch 1: Train Loss=3.2940, Val Loss=1.4062
Epoch 2/15: 100%|██████████████████| 98/98 [02:42<00:00,  1.66s/it, Loss=1.1639]
Epoch 2: Train Loss=1.3860, Val Loss=1.3878
Epoch 3/15:   0%|                                        | 0/98 [00:00<?, ?it/s]

In [ ]:
# !python /kaggle/working/RCS_FUSIONDATA/find_similar_img.py \
# --query_image /kaggle/input/amazon-product/images/1397458135.jpg \
# --df_path /kaggle/working/RCS_FUSIONDATA/data/amazonproduct/val.csv \
# --k 10

In [ ]:
# Setup and imports
import sys, os, pandas as pd, torch, numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from torchvision import transforms
import warnings
warnings.filterwarnings('ignore')

# Add project path
PROJECT_PATH = '/kaggle/working/RCS_FUSIONDATA'
sys.path.append(PROJECT_PATH)

from model import CAMRec
from datahelper import AmazonReviewDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Đã thêm đường dẫn dự án vào sys.path.
Using device: cuda


In [ ]:
class RecommendationSystem:
    def __init__(self, data_dir, model_path, base_dir):
        self.data_dir = data_dir
        self.model_path = model_path
        self.base_dir = base_dir
        self.tokenizer = AutoTokenizer.from_pretrained('roberta-base')
        self.img_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        self._load_data()
        self._create_mappings()
        self._load_model()
    
    def _load_data(self):
        """Load and combine datasets"""
        print("📊 Loading data...")
        train_df = pd.read_csv(f"{self.data_dir}train.csv")
        val_df = pd.read_csv(f"{self.data_dir}val.csv")
        test_df = pd.read_csv(f"{self.data_dir}test.csv")
        
        self.df = pd.concat([train_df, val_df, test_df], ignore_index=True)
        self.df["file_path"] = self.df["file_path"].apply(lambda x: str(Path(".") / x))
        print(f"✓ Records: {len(self.df):,}, Users: {self.df['reviewerID'].nunique():,}, Items: {self.df['asin'].nunique():,}")
    
    def _create_mappings(self):
        """Create ID mappings"""
        print("🔑 Creating mappings...")
        self.users = {u:i for i,u in enumerate(self.df['reviewerID'].astype(str).unique())}
        self.items = {a:i for i,a in enumerate(self.df['asin'].astype(str).unique())}
        print(f"✓ Users: {len(self.users):,}, Items: {len(self.items):,}")
    
    def _load_model(self):
        """Load model"""
        print("🤖 Loading model...")
        self.model = CAMRec(
            n_users=len(self.users), n_items=len(self.items),
            user_dim=128, item_dim=128, proj_dim=256, heads=4
        ).to(device)
        
        if Path(self.model_path).exists():
            try:
                checkpoint = torch.load(self.model_path, map_location=device)
                self.model.load_state_dict(checkpoint, strict=False)
                print("✓ Model loaded")
            except:
                print("⚠ Using random weights")
        else:
            print("⚠ No checkpoint found, using random weights")
        
        self.model.eval()
    
    def select_user(self, min_history=5):
        """Select random user with history"""
        user_counts = self.df.groupby('reviewerID').size()
        eligible_users = user_counts[user_counts > min_history].index.tolist()
        self.selected_user = np.random.choice(eligible_users)
        self.user_idx = self.users[str(self.selected_user)]
        print(f"👤 Selected user: {self.selected_user} (purchases: {user_counts[self.selected_user]})")
        return self.selected_user
    
    def show_history(self, n_show=6):
        """Show user purchase history"""
        history = self.df[self.df['reviewerID'] == self.selected_user].sort_values('overall', ascending=False)
        
        print(f"📜 Purchase history for {self.selected_user}:")
        for _, row in history.head(5).iterrows():
            print(f"  {row['asin']}: {row.get('title', 'N/A')[:50]}... Rating: {row['overall']}")
        
        # Plot images
        fig, axes = plt.subplots(2, 3, figsize=(12, 8))
        axes = axes.flatten()
        
        for i, (_, row) in enumerate(history.head(n_show).iterrows()):
            try:
                img = Image.open(f"{self.base_dir}{row['file_path']}").convert('RGB')
                axes[i].imshow(img)
                axes[i].set_title(f"{row.get('title', 'N/A')[:20]}...\n★{row['overall']}")
                axes[i].axis('off')
            except:
                axes[i].text(0.5, 0.5, 'No image', ha='center', va='center')
                axes[i].axis('off')
        
        for i in range(n_show, len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.suptitle(f'Purchase History - User {self.selected_user}', y=1.02)
        plt.show()
        
        return history
    
    def predict_ratings(self, max_items=5000, batch_size=64):
        """Predict ratings for unpurchased items"""
        print(f"🎯 Predicting ratings for {max_items:,} items...")
        
        # Get unpurchased items
        purchased = set(self.df[self.df['reviewerID'] == self.selected_user]['asin'].astype(str))
        unpurchased = [item for item in self.df['asin'].astype(str).unique() if item not in purchased][:max_items]
        
        predictions = []
        
        with torch.no_grad():
            for i in range(0, len(unpurchased), batch_size):
                batch_items = unpurchased[i:i+batch_size]
                batch_data = {'user_idx': [], 'item_idx': [], 'input_ids': [], 
                             'attention_mask': [], 'image': [], 'price': [], 'meta': []}
                
                for asin in batch_items:
                    try:
                        item_info = self.df[self.df['asin'].astype(str) == asin].iloc[0]
                        
                        # Text
                        text = str(item_info.get('description', ''))[:128]  # Shorter text
                        tokens = self.tokenizer(text, padding='max_length', truncation=True, 
                                              max_length=64, return_tensors='pt')  # Shorter sequence
                        
                        # Image
                        img_path = f"{self.base_dir}{item_info['file_path']}"
                        if Path(img_path).exists():
                            img = Image.open(img_path).convert('RGB')
                            img_tensor = self.img_transform(img)
                        else:
                            img_tensor = torch.zeros(3, 224, 224)
                        
                        batch_data['user_idx'].append(self.user_idx)
                        batch_data['item_idx'].append(self.items[asin])
                        batch_data['input_ids'].append(tokens['input_ids'].squeeze(0))
                        batch_data['attention_mask'].append(tokens['attention_mask'].squeeze(0))
                        batch_data['image'].append(img_tensor)
                        batch_data['price'].append(float(item_info.get('price', 0)))
                        batch_data['meta'].append({
                            'asin': asin,
                            'title': item_info.get('title', 'N/A'),
                            'file_path': item_info['file_path']
                        })
                    except:
                        continue
                
                if not batch_data['user_idx']:
                    continue
                
                # Predict batch
                batch = {
                    'user_idx': torch.tensor(batch_data['user_idx']).to(device),
                    'item_idx': torch.tensor(batch_data['item_idx']).to(device),
                    'input_ids': torch.stack(batch_data['input_ids']).to(device),
                    'attention_mask': torch.stack(batch_data['attention_mask']).to(device),
                    'image': torch.stack(batch_data['image']).to(device),
                    'rating': torch.zeros(len(batch_data['user_idx'])).to(device),
                    'price': torch.tensor(batch_data['price']).to(device)
                }
                
                try:
                    pred_ratings = self.model(batch).cpu().numpy()
                    for j, rating in enumerate(pred_ratings):
                        meta = batch_data['meta'][j]
                        predictions.append({**meta, 'predicted_rating': float(rating)})
                except:
                    continue
                
                if (i + batch_size) % 500 == 0:
                    print(f"  ✓ Processed {min(i + batch_size, len(unpurchased)):,}/{len(unpurchased):,}")
        
        print(f"✓ Predictions: {len(predictions):,}")
        return predictions
    
    def get_recommendations(self, predictions, top_k=10):
        """Get top recommendations"""
        recommendations = sorted(predictions, key=lambda x: x['predicted_rating'], reverse=True)[:top_k]
        
        print(f"\n⭐ TOP {top_k} RECOMMENDATIONS:")
        for i, rec in enumerate(recommendations, 1):
            print(f"{i}. {rec['title'][:50]}... (★{rec['predicted_rating']:.2f})")
        
        # Plot recommendations
        fig, axes = plt.subplots(2, 5, figsize=(16, 6))
        axes = axes.flatten()
        
        for i, rec in enumerate(recommendations):
            try:
                img = Image.open(f"{self.base_dir}{rec['file_path']}").convert('RGB')
                axes[i].imshow(img)
                axes[i].set_title(f"#{i+1}: {rec['title'][:15]}...\n★{rec['predicted_rating']:.2f}")
                axes[i].axis('off')
            except:
                axes[i].text(0.5, 0.5, 'No image', ha='center', va='center')
                axes[i].axis('off')
        
        plt.tight_layout()
        plt.suptitle(f'Recommendations for User {self.selected_user}', y=1.02)
        plt.show()
        
        return recommendations
    
    def run_pipeline(self):
        """Run complete recommendation pipeline"""
        print("🚀 Starting recommendation pipeline\n")
        
        # Select user and show history
        self.select_user()
        history = self.show_history()
        
        # Predict and recommend
        predictions = self.predict_ratings()
        recommendations = self.get_recommendations(predictions)
        
        print("\n✅ Pipeline completed!")
        return recommendations


🚀 STARTING RECOMMENDATION PIPELINE

📊 LOADING DATA...
✓ Total records: 35,733
✓ Number of users: 16,144
✓ Number of items: 13,009
✓ Columns: ['asin', 'title', 'price', 'categories', 'description', 'imUrl', 'reviewerID', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime', 'file_path']

🔑 CREATING MAPPINGS...
✓ User mapping: 16,144 users
✓ Item mapping: 13,009 items

🤖 LOADING MODEL...
⚠ Error in pipeline: 'module' object is not callable
  Please check the logs for more details.


In [ ]:
# Run recommendation system
recommender = RecommendationSystem(
    data_dir="/kaggle/working/RCS_FUSIONDATA/data/amazonproduct/",
    model_path="/kaggle/working/mlp_camrec_model.pth",
    base_dir="/kaggle/working/RCS_FUSIONDATA/"
)

recommendations = recommender.run_pipeline()